# 04 — Multistage / 04 Multistage

**Chapter 20 — File 3 of 3 / 第20章 — 第3个文件（共3个）**

---

## Summary / 总结

This script demonstrates **Load pre-trained model and tokenizer for re-ranking**.

本脚本演示 **Load pre-trained model and tokenizer for re-ranking**。

---
## Step 1 — Step 1

In [ ]:
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import faiss
import numpy as np
import torch

dense_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
dense_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

---
## Step 2 — Load pre-trained model and tokenizer for re-ranking

In [ ]:
reranker_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2")

def generate_embedding(text):
    """Generate dense vector using mean pooling"""
    inputs = dense_tokenizer(text, padding=True, truncation=True, return_tensors="pt",
                             max_length=512)
    with torch.no_grad():
        outputs = dense_model(**inputs)

    attention_mask = inputs["attention_mask"]
    embeddings = outputs.last_hidden_state

    expanded_mask = attention_mask.unsqueeze(-1).expand(embeddings.shape).float()
    sum_embeddings = torch.sum(embeddings * expanded_mask, axis=1)
    sum_mask = torch.clamp(expanded_mask.sum(axis=1), min=1e-9)
    mean_embeddings = sum_embeddings / sum_mask
    return mean_embeddings.cpu().numpy()

def hybrid_retrieval(query, k=3, alpha=0.5):
    """Hybrid retrieval: Use both the BM25 and L2 index on FAISS"""

---
## Step 3 — Sparse score of each document with BM25

In [ ]:
tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

---
## Step 4 — Normalize BM25 scores to [0,1] unless all elements are zero

In [ ]:
if max(bm25_scores) > 0:
        bm25_scores = bm25_scores / max(bm25_scores)

---
## Step 5 — Sort all documents according to L2 distance to query

In [ ]:
query_embedding = generate_embedding(query)
    distances, indices = index.search(query_embedding, len(documents))

---
## Step 6 — Dense score: 1/distance as similarity metric, then normalize to [0,1]

In [ ]:
eps = 1e-5  # a small value to prevent division by zero
    dense_scores = 1 / (eps + np.array(distances[0]))
    dense_scores = dense_scores / max(dense_scores)

---
## Step 7 — Combine scores = affine combination of sparse and dense scores

In [ ]:
combined_scores = alpha * dense_scores + (1 - alpha) * bm25_scores

---
## Step 8 — Get top-k documents

In [ ]:
top_indices = np.argsort(combined_scores)[::-1][:k]
    results = [(documents[idx], combined_scores[idx]) for idx in top_indices]
    return results

def rerank(query, documents, top_k=3):
    """Sort documents by the reranker model and select top-k"""

---
## Step 9 — Prepare inputs for the re-ranker

In [ ]:
pairs = [[query, doc] for doc in documents]
    features = reranker_tokenizer(pairs, padding=True, truncation=True,
                                  return_tensors="pt")

---
## Step 10 — Get re-ranking scores

In [ ]:
with torch.no_grad():
        scores = reranker_model(**features).logits.squeeze(-1).cpu().numpy()

---
## Step 11 — Sort documents by score, then pick top-k

In [ ]:
ranked_indices = np.argsort(scores)[::-1][:top_k]
    reranked_docs = [(documents[idx], float(scores[idx])) for idx in ranked_indices]
    return reranked_docs

def multi_stage_retrieval(query, documents, initial_k=5, final_k=3):
    """Multi-stage retrieval: Hybrid retrievel to shortlist documents, then pick
    with a reranker
    """

---
## Step 12 — Stage 1: Initial retrieval using hybrid method

In [ ]:
initial_results = hybrid_retrieval(query, k=initial_k)
    initial_docs = [doc for doc, _ in initial_results]

---
## Step 13 — Stage 2: Re-ranking

In [ ]:
reranked_results = rerank(query, initial_docs, top_k=final_k)
    return reranked_results

---
## Step 14 — Sample document collection

In [ ]:
documents = [
    "Transformers use self-attention mechanisms to process input sequences in "
        "parallel, making them efficient for long sequences.",
    "The attention mechanism in transformers allows the model to focus on different "
        "parts of the input sequence when generating each output element.",
    "Transformer models have a fixed context length determined by the positional "
        "encoding and self-attention mechanisms.",
    "To handle sequences longer than the context length, transformers can use "
        "techniques like sliding windows or hierarchical processing.",
    "Recurrent Neural Networks (RNNs) process sequences sequentially, which can be "
        "inefficient for long sequences.",
    "Long Short-Term Memory (LSTM) networks are a type of RNN designed to handle "
        "long-term dependencies in sequences.",
    "The Transformer architecture was introduced in the paper 'Attention Is All "
        "You Need' by Vaswani et al.",
    "BERT (Bidirectional Encoder Representations from Transformers) is a "
        "transformer-based model designed for understanding the context of words.",
    "GPT (Generative Pre-trained Transformer) is a transformer-based model designed "
        "for natural language generation.",
    "Transformer-XL extends the context length of transformers by using a "
        "segment-level recurrence mechanism."
]

---
## Step 15 — Prepare for sparse retrieval (BM25)

In [ ]:
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

---
## Step 16 — Prepare for dense retrieval (FAISS)

In [ ]:
document_embeddings = generate_embedding(documents)
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings)

---
## Step 17 — Example query

In [ ]:
query = "How do transformers handle long sequences?"
results = multi_stage_retrieval(query, documents)
print(f"Query: {query}")
print("Re-ranked Results:")
for i, (doc, score) in enumerate(results):
    print(f"Document {i+1} (Score: {score:.4f}):")
    print(doc)
    print()

---
## Learning Notes / 学习笔记

- **概念**: Load pre-trained model and tokenizer for re-ranking 是机器学习中的常用技术。  
  *Load pre-trained model and tokenizer for re-ranking is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Multistage / 04 Multistage
# Complete Code / 完整代码
# ===============================

from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import faiss
import numpy as np
import torch

dense_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
dense_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Load pre-trained model and tokenizer for re-ranking
reranker_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2")

def generate_embedding(text):
    """Generate dense vector using mean pooling"""
    inputs = dense_tokenizer(text, padding=True, truncation=True, return_tensors="pt",
                             max_length=512)
    with torch.no_grad():
        outputs = dense_model(**inputs)

    attention_mask = inputs["attention_mask"]
    embeddings = outputs.last_hidden_state

    expanded_mask = attention_mask.unsqueeze(-1).expand(embeddings.shape).float()
    sum_embeddings = torch.sum(embeddings * expanded_mask, axis=1)
    sum_mask = torch.clamp(expanded_mask.sum(axis=1), min=1e-9)
    mean_embeddings = sum_embeddings / sum_mask
    return mean_embeddings.cpu().numpy()

def hybrid_retrieval(query, k=3, alpha=0.5):
    """Hybrid retrieval: Use both the BM25 and L2 index on FAISS"""
    # Sparse score of each document with BM25
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Normalize BM25 scores to [0,1] unless all elements are zero
    if max(bm25_scores) > 0:
        bm25_scores = bm25_scores / max(bm25_scores)

    # Sort all documents according to L2 distance to query
    query_embedding = generate_embedding(query)
    distances, indices = index.search(query_embedding, len(documents))

    # Dense score: 1/distance as similarity metric, then normalize to [0,1]
    eps = 1e-5  # a small value to prevent division by zero
    dense_scores = 1 / (eps + np.array(distances[0]))
    dense_scores = dense_scores / max(dense_scores)

    # Combine scores = affine combination of sparse and dense scores
    combined_scores = alpha * dense_scores + (1 - alpha) * bm25_scores

    # Get top-k documents
    top_indices = np.argsort(combined_scores)[::-1][:k]
    results = [(documents[idx], combined_scores[idx]) for idx in top_indices]
    return results

def rerank(query, documents, top_k=3):
    """Sort documents by the reranker model and select top-k"""
    # Prepare inputs for the re-ranker
    pairs = [[query, doc] for doc in documents]
    features = reranker_tokenizer(pairs, padding=True, truncation=True,
                                  return_tensors="pt")
    # Get re-ranking scores
    with torch.no_grad():
        scores = reranker_model(**features).logits.squeeze(-1).cpu().numpy()
    # Sort documents by score, then pick top-k
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    reranked_docs = [(documents[idx], float(scores[idx])) for idx in ranked_indices]
    return reranked_docs

def multi_stage_retrieval(query, documents, initial_k=5, final_k=3):
    """Multi-stage retrieval: Hybrid retrievel to shortlist documents, then pick
    with a reranker
    """
    # Stage 1: Initial retrieval using hybrid method
    initial_results = hybrid_retrieval(query, k=initial_k)
    initial_docs = [doc for doc, _ in initial_results]
    # Stage 2: Re-ranking
    reranked_results = rerank(query, initial_docs, top_k=final_k)
    return reranked_results

# Sample document collection
documents = [
    "Transformers use self-attention mechanisms to process input sequences in "
        "parallel, making them efficient for long sequences.",
    "The attention mechanism in transformers allows the model to focus on different "
        "parts of the input sequence when generating each output element.",
    "Transformer models have a fixed context length determined by the positional "
        "encoding and self-attention mechanisms.",
    "To handle sequences longer than the context length, transformers can use "
        "techniques like sliding windows or hierarchical processing.",
    "Recurrent Neural Networks (RNNs) process sequences sequentially, which can be "
        "inefficient for long sequences.",
    "Long Short-Term Memory (LSTM) networks are a type of RNN designed to handle "
        "long-term dependencies in sequences.",
    "The Transformer architecture was introduced in the paper 'Attention Is All "
        "You Need' by Vaswani et al.",
    "BERT (Bidirectional Encoder Representations from Transformers) is a "
        "transformer-based model designed for understanding the context of words.",
    "GPT (Generative Pre-trained Transformer) is a transformer-based model designed "
        "for natural language generation.",
    "Transformer-XL extends the context length of transformers by using a "
        "segment-level recurrence mechanism."
]

# Prepare for sparse retrieval (BM25)
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

# Prepare for dense retrieval (FAISS)
document_embeddings = generate_embedding(documents)
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings)

# Example query
query = "How do transformers handle long sequences?"
results = multi_stage_retrieval(query, documents)
print(f"Query: {query}")
print("Re-ranked Results:")
for i, (doc, score) in enumerate(results):
    print(f"Document {i+1} (Score: {score:.4f}):")
    print(doc)
    print()